# 05. LightGBM Baseline

This notebook will be implemented in the next project stage.

In [1]:
from pathlib import Path
import sys

import pandas as pd

## Environment Setup

In [2]:
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Project root configured.")

PROCESSED_DIR = PROJECT_ROOT / "data/processed"
print("Processed directory configured.")

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
EXPERIMENTS_DIR = ARTIFACTS_DIR / "experiments"

EXPERIMENT_ID = "lightgbm_v0_raw_minimal_cv5"
RUN_TRAINING = True

Project root configured.
Processed directory configured.


In [ ]:
from src.churn_ml.features import load_dataset
from src.churn_ml.modeling import (
    ExperimentConfig,
    load_experiment,
    save_experiment_result,
    build_experiment_summary,
)
from src.churn_ml.models.lightgbm import (
    LightGBMConfig,
    run_lightgbm_experiment,
)
from src.churn_ml.metrics import (
    evaluate_cross_fitted_thresholds,
    optimize_threshold_by_folds,
    summarize_threshold_plateau,
)

c:\Users\pbori\Documents\Courses\Neoversity\ML Foundations\neoversity-ml-final\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Dataset Loading

In [4]:
dataset = load_dataset(
    version="v0_raw_minimal",
    processed_dir=PROCESSED_DIR,
)

print(f"Dataset version: {dataset.version}")
print(f"Train shape:     {dataset.X_train.shape}")
print(f"Target shape:    {dataset.y_train.shape}")
print(f"Test shape:      {dataset.X_test.shape}")

Dataset version: v0_raw_minimal
Train shape:     (10000, 205)
Target shape:    (10000,)
Test shape:      (2500, 205)


## LightGBM Baseline

In [5]:
lightgbm_config = LightGBMConfig.default()
experiment_config = ExperimentConfig.default()

if RUN_TRAINING:
    lightgbm_experiment = run_lightgbm_experiment(
        dataset=dataset,
        config=experiment_config,
        model_config=lightgbm_config,
        experiment_id=EXPERIMENT_ID,
    )

    save_experiment_result(
        result=lightgbm_experiment.result,
        experiments_dir=EXPERIMENTS_DIR,
        fold_metrics=lightgbm_experiment.fold_metrics,
        oof_predictions=lightgbm_experiment.oof_predictions,
        test_predictions=lightgbm_experiment.test_predictions,
        overwrite=True,
    )
else:
    lightgbm_experiment = load_experiment(
        experiment_id=EXPERIMENT_ID,
        experiments_dir=EXPERIMENTS_DIR,
    )

Starting LightGBM cross-validation: 5 folds, 10,000 rows, 205 features


LightGBM CV:   0%|          | 0/5 [00:00<?, ?fold/s]c:\Users\pbori\Documents\Courses\Neoversity\ML Foundations\neoversity-ml-final\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
LightGBM CV:  20%|██        | 1/5 [00:02<00:08,  2.16s/fold, balanced_accuracy=0.7479, best_iteration=133, roc_auc=0.9591]c:\Users\pbori\Documents\Courses\Neoversity\ML Foundations\neoversity-ml-final\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
LightGBM CV:  40%|████      | 2/5 [00:04<00:06,  2.14s/fold, balanced_accuracy=0.7587, best_iteration=132, roc_auc=0.9495]c:\Users\pbori\Documents\Courses\Neoversity\ML Foundations\neoversity-ml-final\.ve

## Experiment Results

In [6]:
display(lightgbm_experiment.fold_metrics)

display(
    pd.Series(
        lightgbm_experiment.result.metrics,
        name="value",
    ).to_frame()
)

,fold,balanced_accuracy,roc_auc,average_precision,log_loss,decision_threshold,training_time_seconds,best_iteration
0,1,0.747889,0.959115,0.800551,0.175670,0.5,2.042835,133
1,2,0.758711,0.949478,0.781379,0.184650,0.5,1.995591,132
2,3,0.792141,0.955905,0.800543,0.172173,0.5,2.364535,173
3,4,0.767427,0.954686,0.781288,0.180142,0.5,2.086108,132
4,5,0.750951,0.951154,0.764824,0.185793,0.5,2.233726,125


,value
balanced_accuracy,0.763424
roc_auc,0.953695
average_precision,0.784128
log_loss,0.179686
decision_threshold,0.500000
optimized_threshold_oof,0.128000
optimized_balanced_accuracy_oof,0.881303
balanced_accuracy_mean,0.763424
balanced_accuracy_std,0.017747
roc_auc_mean,0.954067


In [7]:
summary = build_experiment_summary(lightgbm_experiment.result)

display(summary.style.format({"value": "{:.4f}"}))

,metric,value
0,Balanced Accuracy @ default threshold,0.7634
1,Optimized OOF Balanced Accuracy,0.8813
2,Optimized OOF threshold,0.1280
3,ROC-AUC,0.9537
4,Average Precision,0.7841
5,Log Loss,0.1797
6,"Training time, minutes",0.1896


In [8]:
fold_columns = [
    "fold",
    "balanced_accuracy",
    "roc_auc",
    "average_precision",
    "log_loss",
    "best_iteration",
    "training_time_seconds",
]

display(
    lightgbm_experiment.fold_metrics[fold_columns].style.format(
        {
            "balanced_accuracy": "{:.4f}",
            "roc_auc": "{:.4f}",
            "average_precision": "{:.4f}",
            "log_loss": "{:.4f}",
            "training_time_seconds": "{:.1f}",
        }
    )
)

,fold,balanced_accuracy,roc_auc,average_precision,log_loss,best_iteration,training_time_seconds
0,1,0.7479,0.9591,0.8006,0.1757,133,2.0
1,2,0.7587,0.9495,0.7814,0.1847,132,2.0
2,3,0.7921,0.9559,0.8005,0.1722,173,2.4
3,4,0.7674,0.9547,0.7813,0.1801,132,2.1
4,5,0.7510,0.9512,0.7648,0.1858,125,2.2


## Threshold Stability

In [9]:
fold_thresholds = optimize_threshold_by_folds(
    fold_metrics=lightgbm_experiment.fold_metrics,
    oof_predictions=lightgbm_experiment.oof_predictions,
)

display(
    fold_thresholds.style.format(
        {
            "optimized_threshold": "{:.3f}",
            "optimized_balanced_accuracy": "{:.4f}",
        }
    )
)

print(f"Threshold mean: {fold_thresholds['optimized_threshold'].mean():.3f}")

print(f"Threshold std:  {fold_thresholds['optimized_threshold'].std(ddof=1):.3f}")

,fold,optimized_threshold,optimized_balanced_accuracy
0,1,0.086,0.8954
1,2,0.084,0.8686
2,3,0.102,0.8814
3,4,0.132,0.8892
4,5,0.162,0.8858


Threshold mean: 0.113
Threshold std:  0.033


## Cross-Fitted Threshold Evaluation

In [10]:
cross_fitted_result = evaluate_cross_fitted_thresholds(
    lightgbm_experiment.oof_predictions
)

cross_fitted_fold_metrics = cross_fitted_result.fold_metrics

display(
    cross_fitted_fold_metrics.style.format(
        {
            "threshold_cross_fitted": "{:.3f}",
            "balanced_accuracy_cross_fitted": "{:.4f}",
            "threshold_fold_optimal": "{:.3f}",
            "balanced_accuracy_fold_optimal": "{:.4f}",
            "balanced_accuracy_regret": "{:.4f}",
        }
    )
)

,fold,threshold_cross_fitted,balanced_accuracy_cross_fitted,threshold_fold_optimal,balanced_accuracy_fold_optimal,balanced_accuracy_regret
0,1,0.128,0.8914,0.086,0.8954,0.0039
1,2,0.128,0.8673,0.084,0.8686,0.0013
2,3,0.132,0.8741,0.102,0.8814,0.0073
3,4,0.128,0.8869,0.132,0.8892,0.0023
4,5,0.128,0.8835,0.162,0.8858,0.0023


In [11]:
optimized_oof_balanced_accuracy = lightgbm_experiment.result.metrics[
    "optimized_balanced_accuracy_oof"
]

optimism_gap = optimized_oof_balanced_accuracy - cross_fitted_result.balanced_accuracy

print(
    f"Cross-fitted OOF Balanced Accuracy: {cross_fitted_result.balanced_accuracy:.4f}"
)
print(
    "Mean cross-fitted threshold:         "
    f"{cross_fitted_fold_metrics['threshold_cross_fitted'].mean():.3f}"
)
print(
    "Cross-fitted threshold std:          "
    f"{cross_fitted_fold_metrics['threshold_cross_fitted'].std(ddof=1):.3f}"
)
print(
    "Mean Balanced Accuracy regret:       "
    f"{cross_fitted_fold_metrics['balanced_accuracy_regret'].mean():.4f}"
)
print(f"OOF threshold-selection optimism:    {optimism_gap:.4f}")

Cross-fitted OOF Balanced Accuracy: 0.8807
Mean cross-fitted threshold:         0.129
Cross-fitted threshold std:          0.002
Mean Balanced Accuracy regret:       0.0034
OOF threshold-selection optimism:    0.0007


## Threshold Plateau

In [12]:
threshold_plateau = summarize_threshold_plateau(
    y_true=lightgbm_experiment.oof_predictions["target"],
    probabilities=lightgbm_experiment.oof_predictions["probability"].to_numpy(),
    tolerance=0.001,
)

threshold_plateau_summary = pd.DataFrame(
    [
        {
            "best_threshold": threshold_plateau.best_threshold,
            "best_balanced_accuracy": (threshold_plateau.best_balanced_accuracy),
            "lower_threshold": threshold_plateau.lower_threshold,
            "upper_threshold": threshold_plateau.upper_threshold,
            "midpoint_threshold": (threshold_plateau.midpoint_threshold),
            "plateau_width": threshold_plateau.width,
            "tolerance": threshold_plateau.tolerance,
        }
    ]
)

display(
    threshold_plateau_summary.style.format(
        {
            "best_threshold": "{:.3f}",
            "best_balanced_accuracy": "{:.4f}",
            "lower_threshold": "{:.3f}",
            "upper_threshold": "{:.3f}",
            "midpoint_threshold": "{:.3f}",
            "plateau_width": "{:.3f}",
            "tolerance": "{:.4f}",
        }
    )
)

,best_threshold,best_balanced_accuracy,lower_threshold,upper_threshold,midpoint_threshold,plateau_width,tolerance
0,0.128,0.8813,0.126,0.132,0.129,0.006,0.0010


In [13]:
fold_plateau_records = []

for fold_number in sorted(lightgbm_experiment.oof_predictions["fold"].unique()):
    fold_data = lightgbm_experiment.oof_predictions.loc[
        lightgbm_experiment.oof_predictions["fold"] == fold_number
    ]

    fold_plateau = summarize_threshold_plateau(
        y_true=fold_data["target"],
        probabilities=fold_data["probability"].to_numpy(),
        tolerance=0.001,
    )

    fold_plateau_records.append(
        {
            "fold": int(fold_number),
            "best_threshold": fold_plateau.best_threshold,
            "lower_threshold": fold_plateau.lower_threshold,
            "upper_threshold": fold_plateau.upper_threshold,
            "midpoint_threshold": (fold_plateau.midpoint_threshold),
            "plateau_width": fold_plateau.width,
            "best_balanced_accuracy": (fold_plateau.best_balanced_accuracy),
        }
    )

fold_plateaus = pd.DataFrame(fold_plateau_records)

display(
    fold_plateaus.style.format(
        {
            "best_threshold": "{:.3f}",
            "lower_threshold": "{:.3f}",
            "upper_threshold": "{:.3f}",
            "midpoint_threshold": "{:.3f}",
            "plateau_width": "{:.3f}",
            "best_balanced_accuracy": "{:.4f}",
        }
    )
)

,fold,best_threshold,lower_threshold,upper_threshold,midpoint_threshold,plateau_width,best_balanced_accuracy
0,1,0.086,0.085,0.093,0.089,0.008,0.8954
1,2,0.084,0.083,0.084,0.083,0.001,0.8686
2,3,0.102,0.100,0.104,0.102,0.004,0.8814
3,4,0.132,0.132,0.144,0.138,0.012,0.8892
4,5,0.162,0.158,0.163,0.161,0.005,0.8858


In [14]:
selected_threshold = lightgbm_experiment.result.metrics["optimized_threshold_oof"]

selected_threshold_inside_plateau = (
    threshold_plateau.lower_threshold
    <= selected_threshold
    <= threshold_plateau.upper_threshold
)

print(f"Selected OOF threshold:       {selected_threshold:.3f}")
print(
    "Near-optimal plateau:        "
    f"[{threshold_plateau.lower_threshold:.3f}, "
    f"{threshold_plateau.upper_threshold:.3f}]"
)
print(f"Selected threshold in plateau: {selected_threshold_inside_plateau}")

Selected OOF threshold:       0.128
Near-optimal plateau:        [0.126, 0.132]
Selected threshold in plateau: True


## Baseline Conclusion

In [15]:
final_threshold_summary = pd.DataFrame(
    [
        {
            "metric": "Balanced Accuracy @ 0.5",
            "value": lightgbm_experiment.result.metrics["balanced_accuracy"],
        },
        {
            "metric": "Optimized OOF Balanced Accuracy",
            "value": lightgbm_experiment.result.metrics[
                "optimized_balanced_accuracy_oof"
            ],
        },
        {
            "metric": "Cross-fitted Balanced Accuracy",
            "value": cross_fitted_result.balanced_accuracy,
        },
        {
            "metric": "Selected threshold",
            "value": selected_threshold,
        },
        {
            "metric": "Cross-fitted threshold mean",
            "value": cross_fitted_fold_metrics["threshold_cross_fitted"].mean(),
        },
        {
            "metric": "Threshold plateau lower bound",
            "value": threshold_plateau.lower_threshold,
        },
        {
            "metric": "Threshold plateau upper bound",
            "value": threshold_plateau.upper_threshold,
        },
        {
            "metric": "ROC-AUC",
            "value": lightgbm_experiment.result.metrics["roc_auc"],
        },
        {
            "metric": "Average Precision",
            "value": lightgbm_experiment.result.metrics["average_precision"],
        },
    ]
)

display(final_threshold_summary.style.format({"value": "{:.4f}"}))

,metric,value
0,Balanced Accuracy @ 0.5,0.7634
1,Optimized OOF Balanced Accuracy,0.8813
2,Cross-fitted Balanced Accuracy,0.8807
3,Selected threshold,0.1280
4,Cross-fitted threshold mean,0.1288
5,Threshold plateau lower bound,0.1260
6,Threshold plateau upper bound,0.1320
7,ROC-AUC,0.9537
8,Average Precision,0.7841


In [16]:
expected_test_predictions = (
    lightgbm_experiment.test_predictions["probability"] >= selected_threshold
).astype(int)

assert expected_test_predictions.equals(
    lightgbm_experiment.test_predictions["prediction_optimized_oof"]
)

test_positive_rate = expected_test_predictions.mean()

print(f"Selected threshold:       {selected_threshold:.3f}")
print(f"Test positive rate:       {test_positive_rate:.2%}")
print(f"Predicted positive rows:  {expected_test_predictions.sum():,}")
print(f"Total test rows:          {len(expected_test_predictions):,}")

Selected threshold:       0.128
Test positive rate:       22.68%
Predicted positive rows:  567
Total test rows:          2,500


In [17]:
experiment_dir = EXPERIMENTS_DIR / EXPERIMENT_ID
experiment_dir.mkdir(parents=True, exist_ok=True)

cross_fitted_fold_metrics.to_csv(
    experiment_dir / "cross_fitted_threshold_metrics.csv",
    index=False,
)

fold_plateaus.to_csv(
    experiment_dir / "threshold_plateaus_by_fold.csv",
    index=False,
)

final_threshold_summary.to_csv(
    experiment_dir / "threshold_summary.csv",
    index=False,
)

print(f"Threshold diagnostics saved to: {experiment_dir}")

Threshold diagnostics saved to: c:\Users\pbori\Documents\Courses\Neoversity\ML Foundations\neoversity-ml-final\artifacts\experiments\lightgbm_v0_raw_minimal_cv5


## Feature Importance

Gain-based feature importance is averaged across the five fitted LightGBM folds. Gain measures the total reduction in the training objective attributed to splits using each feature.

In [18]:
feature_importance_by_fold = []

for fold_number, model in enumerate(
    lightgbm_experiment.fitted_models,
    start=1,
):
    fold_importance = pd.DataFrame(
        {
            "feature": dataset.X_train.columns,
            "importance": model.booster_.feature_importance(importance_type="gain"),
            "fold": fold_number,
        }
    )

    total_importance = fold_importance["importance"].sum()

    if total_importance > 0:
        fold_importance["importance"] /= total_importance

    feature_importance_by_fold.append(fold_importance)

feature_importance_folds = pd.concat(
    feature_importance_by_fold,
    ignore_index=True,
)

feature_importance_summary = (
    feature_importance_folds.groupby("feature", as_index=False)
    .agg(
        importance_mean=("importance", "mean"),
        importance_std=("importance", "std"),
    )
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

display(
    feature_importance_summary.head(30).style.format(
        {
            "importance_mean": "{:.4%}",
            "importance_std": "{:.4%}",
        }
    )
)

,feature,importance_mean,importance_std
0,Var126,26.4637%,1.1425%
1,Var212,13.5913%,0.9988%
2,Var192,6.5678%,0.3094%
3,Var73,5.8379%,0.1963%
4,Var216,5.2087%,1.4695%
5,Var199,4.1416%,0.3443%
6,Var74,3.4795%,0.1877%
7,Var144,3.3468%,0.1338%
8,Var189,2.8621%,0.1401%
9,Var57,2.5936%,0.1318%


In [19]:
feature_importance_summary.to_csv(
    experiment_dir / "feature_importance.csv",
    index=False,
)

print(f"Feature importance saved to: {experiment_dir / 'feature_importance.csv'}")

Feature importance saved to: c:\Users\pbori\Documents\Courses\Neoversity\ML Foundations\neoversity-ml-final\artifacts\experiments\lightgbm_v0_raw_minimal_cv5\feature_importance.csv


## Conclusion

The LightGBM baseline was evaluated using the same five-fold stratified cross-validation protocol as the CatBoost baseline.

The model achieved:

- ROC-AUC of approximately **0.954**
- Average Precision of approximately **0.784**
- Balanced Accuracy of approximately **0.763** at the default threshold of 0.5
- Optimized OOF Balanced Accuracy of approximately **0.881**
- Cross-fitted Balanced Accuracy of approximately **0.881**

The optimized OOF decision threshold was **0.128**. Cross-fitted threshold estimates were highly stable, with a mean of approximately 0.129 and a standard deviation below 0.002. The global threshold also lies inside the near-optimal plateau, indicating that the selected threshold is robust.

Although LightGBM trained substantially faster than CatBoost, it produced lower ROC-AUC, Average Precision, and threshold-optimized Balanced Accuracy. Therefore, CatBoost remains the strongest standalone baseline at this stage.

LightGBM is retained as a useful complementary model because of its computational efficiency and potentially different error structure. Its predictions may later contribute to model blending or stacking.